Thie notebook creates simulations to contextualize the FN and FP rate for the hypothesis "cre of interest is significantly different from minP".
- negative simulations (cre_oi=minP) for quantifying the FP rate.
- positive simulations (cre_oi!=minP) for quantifying FN rate.

# Setup

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

%load_ext autoreload
%autoreload 2

2026-01-14 16:04:39.675342: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-14 16:04:39.678646: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [4]:
data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"

# Creating artificial libraries

In [ ]:
#making up the CREs
spread_gt,spread_hypothesis=scm.activity_spread(
    cell_types=list(scm.SHENDURE_BOUNDS.cells_per_cell_type.keys()),
    minimum=scm.SHENDURE_BOUNDS.min_mpra_umi,
    maximum=scm.SHENDURE_BOUNDS.max_mpra_umi,
    minp_value=scm.SHENDURE_BOUNDS.reference_activity,
    total=100,
    frac_active=0.5,
    ct_specificity=.2)

libraries=[scm.simulate_library(CREs=spread_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [ ]:
spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")

# Creating sim

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=scm.SHENDURE_BOUNDS,
                            ground_truth=spread_gt)

In [ ]:
sim.gamut()

In [ ]:
sim.save()

# Fit orthos

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

In [ ]:
sim.fit_orthos()

In [ ]:
sim.save()

# Wald precompute: sandwich

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

In [ ]:
sim.precompute_wald(cov_method="sandwich")

In [ ]:
sim.save()

# Wald precompute: opg

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

In [ ]:
sim.precompute_wald(cov_method="opg")

In [ ]:
sim.save()

# Hypothesis testing

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

Add a basic hypothesis set

In [ ]:
spread_hypothesis=scm.HypothesisSet.from_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")

In [ ]:
sim.add_hypothesis_set(name="spread",hypotheses=spread_hypothesis)

Run mwu

In [ ]:
sim.mwu("spread")

In [ ]:
sim.save()

In [ ]:
sim.wald("spread")

In [ ]:
sim.wald("spread",cov_method="opg")

In [ ]:
sim.save()

# Summary

In [5]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

scMPRAforge: INFO: 'state.parquet' found for 'pow_sim_2026-01-03', loading.


In [6]:
sim.list_tests()

{'spread': ['mwu', 'wald_opg', 'wald_sandwich']}

In [10]:
sim._classifier_summary(hypothesis_set_name="spread",test_type="mwu")

     comparison_CRE reference_CRE     comparison_cell_type  \
0        inactive_0     reference           Cardiomyocytes   
1        inactive_0     reference  EpiblastPrimitiveStreak   
2        inactive_0     reference       ExEndodermParietal   
3        inactive_0     reference       ExEndodermVisceral   
4        inactive_0     reference       Haematoendothelial   
...             ...           ...                      ...   
5495      active_45     active_45                reference   
5496      active_46     active_46                reference   
5497      active_47     active_47                reference   
5498      active_48     active_48                reference   
5499      active_49     active_49                reference   

          reference_cell_type  meta  test_statistic       p_value  \
0              Cardiomyocytes     0        143989.0  6.477094e-01   
1     EpiblastPrimitiveStreak     0       3068977.5  6.391818e-01   
2          ExEndodermParietal     0       603864

,replicate,auroc,auprc
0,0,0.989646,0.978985
1,1,0.990315,0.980703
2,2,0.984492,0.974943
3,3,0.974656,0.961427
4,4,0.982285,0.967348


In [13]:
sim._all_classifier_summary("spread")

scMPRAforge: INFO: Dropped 7 or 0.1% of tests with NA values for test:wald_opg, rep:0.
scMPRAforge: INFO: Dropped 7 or 0.1% of tests with NA values for test:wald_opg, rep:1.
scMPRAforge: INFO: Dropped 7 or 0.1% of tests with NA values for test:wald_opg, rep:2.
scMPRAforge: INFO: Dropped 7 or 0.1% of tests with NA values for test:wald_opg, rep:3.
scMPRAforge: INFO: Dropped 7 or 0.1% of tests with NA values for test:wald_opg, rep:4.
scMPRAforge: INFO: Dropped 63 or 1.1% of tests with NA values for test:wald_sandwich, rep:0.
scMPRAforge: INFO: Dropped 63 or 1.1% of tests with NA values for test:wald_sandwich, rep:1.
scMPRAforge: INFO: Dropped 63 or 1.1% of tests with NA values for test:wald_sandwich, rep:2.
scMPRAforge: INFO: Dropped 63 or 1.1% of tests with NA values for test:wald_sandwich, rep:3.
scMPRAforge: INFO: Dropped 63 or 1.1% of tests with NA values for test:wald_sandwich, rep:4.


,replicate,auroc,auprc,test
0,0,0.989646,0.978985,mwu
1,1,0.990315,0.980703,mwu
2,2,0.984492,0.974943,mwu
3,3,0.974656,0.961427,mwu
4,4,0.982285,0.967348,mwu
0,0,0.920082,0.858811,wald_opg
1,1,0.920082,0.858811,wald_opg
2,2,0.920082,0.858811,wald_opg
3,3,0.920082,0.858811,wald_opg
4,4,0.920082,0.858811,wald_opg


# Shutdown

In [30]:
client.close()
cluster.close()